# 00 — Colab runtime bootstrap
Connect this notebook to a Google Colab GPU kernel from VS Code, then run cells in order. Edit only the settings cell; do not store credentials here.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU-backed Colab kernel.')
props = torch.cuda.get_device_properties(0)
print('GPU:', props.name)
print(f'Total VRAM: {props.total_memory / 1024**3:.1f} GiB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- SETTINGS: repository plus one-time dataset-license acknowledgement ----
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
ACCEPT_COD10K_NONCOMMERCIAL_LICENSE = False  # Review the official license, then set True.

PROJECT_BRANCH = 'main'
PROJECT_DIR = '/content/cod-ssl'
DINOV3_REPO_DIR = '/content/third_party/dinov3'
VJEPA2_REPO_DIR = '/content/third_party/vjepa2'
WEIGHTS_DIR = '/content/drive/MyDrive/cod-ssl/checkpoints'
DINOV3_WEIGHTS = f'{WEIGHTS_DIR}/dinov3_vitb16.pth'
VJEPA21_WEIGHTS = f'{WEIGHTS_DIR}/vjepa2_1_vitb_dist_vitG_384.pt'
SAMPLE_IMAGE = '/content/cod_ssl_sample_image.png'

In [ ]:
from pathlib import Path
import subprocess
def clone_or_update(url, destination, branch=None):
    target = Path(destination)
    if (target / '.git').is_dir():
        subprocess.run(['git','-C',destination,'fetch','--all','--prune'], check=True)
        if branch:
            subprocess.run(['git','-C',destination,'checkout',branch], check=True)
            subprocess.run(['git','-C',destination,'pull','--ff-only','origin',branch], check=True)
        else: subprocess.run(['git','-C',destination,'pull','--ff-only'], check=True)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        command = ['git','clone'] + (['--branch',branch] if branch else [])
        subprocess.run(command + [url,destination], check=True)
if 'YOUR_USERNAME' in PROJECT_REPO_URL: raise ValueError('Set PROJECT_REPO_URL first.')
clone_or_update(PROJECT_REPO_URL, PROJECT_DIR, PROJECT_BRANCH)
clone_or_update('https://github.com/facebookresearch/dinov3.git', DINOV3_REPO_DIR)
clone_or_update('https://github.com/facebookresearch/vjepa2.git', VJEPA2_REPO_DIR)

In [ ]:
import sys
subprocess.run([sys.executable,'-m','pip','install','-e',f'{PROJECT_DIR}[dev,notebooks]'], check=True)

In [ ]:
# Download/cache/extract the official combined 4,040-image training set and build its manifest.
# Official source: DengPingFan/SINet. COD10K is licensed for non-commercial use.
if not ACCEPT_COD10K_NONCOMMERCIAL_LICENSE:
    raise PermissionError(
        'Review https://github.com/DengPingFan/SINet#9-license, then set '
        'ACCEPT_COD10K_NONCOMMERCIAL_LICENSE=True in the settings cell.'
    )
DATA_ROOT = '/content/drive/MyDrive/cod-ssl/data'
subprocess.run([
    sys.executable, f'{PROJECT_DIR}/scripts/bootstrap_training_data.py',
    '--data-root', DATA_ROOT,
    '--manifest', f'{PROJECT_DIR}/manifests/train_all.csv',
    '--accept-noncommercial-license',
], cwd=PROJECT_DIR, check=True)

In [ ]:
# Download and cache checkpoints. V-JEPA is public; DINOv3 requires your private Meta URL once.
from getpass import getpass
from pathlib import Path
from urllib.request import Request, urlopen
import os

VJEPA21_URL = 'https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt'
Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)

def download_private(url, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + '.part')
    request = Request(url, headers={'User-Agent':'Wget/1.21.4'})
    with urlopen(request) as response, temporary.open('wb') as output:
        total = int(response.headers.get('Content-Length', 0))
        received = 0
        while chunk := response.read(8 * 1024 * 1024):
            output.write(chunk); received += len(chunk)
            if total: print(f'\r{destination.name}: {100 * received / total:.1f}%', end='')
    temporary.replace(destination)
    print(f'\nSaved {destination} ({destination.stat().st_size / 1024**2:.1f} MiB)')

if not Path(VJEPA21_WEIGHTS).is_file():
    print('Downloading public V-JEPA 2.1 ViT-B/16 checkpoint...')
    download_private(VJEPA21_URL, VJEPA21_WEIGHTS)
else: print('Using cached V-JEPA checkpoint:', VJEPA21_WEIGHTS)

if not Path(DINOV3_WEIGHTS).is_file():
    print('DINOv3 requires approved Meta access. Paste the ViT-B/16 LVD-1689M URL from Meta.')
    dinov3_url = getpass('Private DINOv3 checkpoint URL (input hidden): ').strip()
    if not dinov3_url: raise ValueError('A Meta-authorized DINOv3 URL is required on the first run.')
    download_private(dinov3_url, DINOV3_WEIGHTS)
    del dinov3_url
else: print('Using cached DINOv3 checkpoint:', DINOV3_WEIGHTS)

In [ ]:
# Download a public COD illustration for shape-only backbone smoke testing.
# Source: DengPingFan/SINet. This is not training or evaluation data.
SAMPLE_IMAGE_URL = (
    'https://raw.githubusercontent.com/DengPingFan/SINet/'
    'master/Images/CamouflagedTask.png'
)
if not Path(SAMPLE_IMAGE).is_file():
    download_private(SAMPLE_IMAGE_URL, SAMPLE_IMAGE)
else:
    print('Using cached smoke-test image:', SAMPLE_IMAGE)

In [ ]:
import os
os.environ.update({'DINOV3_REPO_DIR':DINOV3_REPO_DIR,'DINOV3_WEIGHTS':DINOV3_WEIGHTS,'VJEPA2_REPO_DIR':VJEPA2_REPO_DIR,'VJEPA21_WEIGHTS':VJEPA21_WEIGHTS})
required = {'project':PROJECT_DIR,'DINOv3 repo':DINOV3_REPO_DIR,'DINOv3 weights':DINOV3_WEIGHTS,'V-JEPA repo':VJEPA2_REPO_DIR,'V-JEPA weights':VJEPA21_WEIGHTS,'sample image':SAMPLE_IMAGE}
missing = [f'{label}: {path}' for label,path in required.items() if not Path(path).exists()]
if missing: raise FileNotFoundError('Missing required paths:\n' + '\n'.join(missing))
print('All configured paths exist.')

In [ ]:
subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/runtime_check.py'],cwd=PROJECT_DIR,check=True)
subprocess.run([sys.executable,'-m','pytest','-q'],cwd=PROJECT_DIR,check=True)

Bootstrap complete. Continue with `01_backbone_feature_smoke_test.ipynb` on the same Colab kernel.